In [55]:
import pandas as pd
import numpy as np
import joblib
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.ensemble import RandomForestRegressor

In [56]:
def load_and_explore_dataset(file_path):
    print("=" * 60)
    print("LOAD AND EXPLORE DATASET")
    print("=" * 60)
    
    pd.set_option('display.float_format', lambda x: f"{x:.2f}")

    df = pd.read_csv(file_path)
    print("Shape of dataset")
    print(df.shape)
    print("\nChecking for missing value:")
    print(df.isnull().sum())
    print("\nFirst five rows:")
    print(df.head())
    print("\nDescriptive statistic:")
    print(df.describe())
    print("\nDataset info:")
    print(df.info())
    print("\nCondition distribution:")
    print(df["Condition"].value_counts())
    print("\nTransmission distribution:")
    print(df["Transmission"].value_counts())

    return df

In [57]:
def preprocessed_data(df):
    print("=" * 60)
    print("PREPROCESSING DATASET")
    print("=" * 60)

    df_preprocessed = df.copy()
    df_preprocessed = df.dropna()

    label_encoder = {}
    df_columns = ['Make', 'Model', 'Condition', 'Transmission']    
    for col in df_columns:
        le = LabelEncoder()
        df_preprocessed[col + "_encoded"] = le.fit_transform(df_preprocessed[col])
        label_encoder[col] = le 
        print(f"\n{col} encoded")
        for i, label in enumerate(le.classes_):
            print(f"{label}: {i}")
    print("preprocessed data shape:", df_preprocessed.shape)

    return df_preprocessed, label_encoder

In [58]:
def feature_data(df_preprocessed):
    print("\n" + "=" * 60)
    print("FEATURE DATASET")
    print("=" * 60)

    feature_columns = ['Year', 'Make_encoded', 'Model_encoded', 'Transmission_encoded', 'Condition_encoded']
    target = ['Price']

    X = df_preprocessed[feature_columns]
    y = df_preprocessed[target]

    print("\nFeature shape:", df_preprocessed[feature_columns].shape)
    print("\nTarget shape:" , df_preprocessed[target].shape)
    print("\nFeature columns:", feature_columns)

    return X, y, feature_columns

In [59]:
def split_data(X,y, test_size=0.2, random_state=42):
    print("\n" + "=" * 60)
    print("SPLITTING DATASET")
    print("=" * 60)

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=test_size, random_state=random_state
    )

    print("\nTraining set size:", X_train.shape[0])
    print("\nTesting set size:", X_test.shape[0])

    
    print("\nTraining set range {:.2f} - {:.2f}".format(
        float(y_train.values.min()), float(y_train.values.max())
    ))
    
    print("\nTesting set range {:.2f} - {:.2f}".format(
         float(y_test.values.min()), float(y_test.values.max())
    ))
    

    return X_train, X_test, y_train, y_test


In [60]:
def scale_feature(X_train, X_test):
    print("\n" + "=" * 60)
    print("SCALE FEATURES DATASET")
    print("=" * 60)

    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.fit_transform(X_test)

    print("\nFeatures scaled successful")
    print("\nTrain scaled features:", X_train_scaled.shape)
    print("\nTest scaled features:", X_test_scaled.shape)

    return X_train_scaled, X_test_scaled, scaler

In [61]:
def train_model(X_train_scaled, y_train, feature_columns):
    print("\n" + "=" * 60)
    print("TRAINING MODEL")
    print("=" * 60)

    model = LinearRegression()
    model.fit(X_train_scaled, y_train)

    print("\nModel train successfully")
    print("\nModel coefficient")

    for feature, coef in zip(feature_columns, model.coef_.ravel()):
        print(f"   {feature}: {coef:.2f}")
    print(f"\nModel intercept: {float(model.intercept_[0]):.2f}")

    return model




In [62]:
def rm_train_model(X_train_scaled, y_train, feature_columns):
    print("\n" + "=" * 60)
    print("RANDOM FOREST TRAINING MODEL")
    print("=" * 60)
    rm_model = RandomForestRegressor(
        n_estimators=100,
        max_depth=15,
        min_samples_split=5,
        min_samples_leaf=2,
        random_state=42,
        n_jobs=-1
    )
    rm_model.fit(X_train_scaled, y_train)

    print("\nModel train successfully")

    feature_importance = sorted(zip(feature_columns, rm_model.feature_importances_),
                                key=lambda x: x[1], reverse=True)

    for feature, importance in feature_importance:
        print(f"  {feature}: {float(importance)}")
        
    return rm_model

In [63]:
def evaluate_model(model, X_train_scaled, X_test_scaled, y_train, y_test):
    print("\n" + "=" * 60)
    print("EVALUAING MODEL")
    print("=" * 60)
    

    #make prediction

    y_train_pred = model.predict(X_train_scaled)
    y_test_pred = model.predict(X_test_scaled)

    #calculate metrics

    train_r2_score = r2_score(y_train, y_train_pred)
    test_r2_score = r2_score(y_test, y_test_pred)

    train_rmse = np.sqrt(mean_squared_error(y_train, y_train_pred))
    test_rmse = np.sqrt(mean_squared_error(y_test, y_test_pred))

    train_mae = mean_absolute_error(y_train, y_train_pred)
    test_mae = mean_absolute_error(y_test, y_test_pred)

    print("\n" + "=" * 60)


    print("MODEL PERFORMANCE")
    print("=" * 60)
    print("TRAINING SET:")
    print(f"  R2 score: {train_r2_score:.4f}")
    print(f"  RMSE: {train_rmse:.4f}")
    print(f"  MAE:  {train_mae:.4f}")

    print("\nTesting set:")
    print(f"  R2 score: {test_r2_score:.4f}")
    print(f"  RMSE: {test_rmse:.4f}")
    print(f"  MAE:  {test_mae:.4f}")

    cv_score = cross_val_score(model, X_train_scaled, y_train, cv=5, scoring='r2')

    print("\nCross validation (5 folds)")
    print(f"  R2 score: {cv_score}")
    print(f"  Cross val mean: {cv_score.mean()}")
    print(f"  Cross val std: {cv_score.std()}")

    metrics = {
        "train_r2_score": train_r2_score,
        "test_r2_score": test_r2_score,
        "train_rmse": train_rmse,
        "test_rmse": test_rmse,
        "train_mae": train_mae,
        "test_mae": test_mae,
        "cv_score": cv_score

    }
        


    return metrics

In [64]:
def rm_evaluate_model(rm_model, X_train_scaled, X_test_scaled, y_train, y_test):
    print("\n" + "=" * 60)
    print("RANDOM FOREST EVALUAING MODEL")
    print("=" * 60)
    

    #make prediction

    y_train_pred = rm_model.predict(X_train_scaled)
    y_test_pred = rm_model.predict(X_test_scaled)

    #calculate metrics

    train_r2_score = r2_score(y_train, y_train_pred)
    test_r2_score = r2_score(y_test, y_test_pred)

    train_rmse = np.sqrt(mean_squared_error(y_train, y_train_pred))
    test_rmse = np.sqrt(mean_squared_error(y_test, y_test_pred))

    train_mae = mean_absolute_error(y_train, y_train_pred)
    test_mae = mean_absolute_error(y_test, y_test_pred)

    print("\n" + "=" * 60)


    print("RANDOM FOREST MODEL PERFORMANCE")
    print("=" * 60)
    print("TRAINING SET:")
    print(f"  R2 score: {train_r2_score:.4f}")
    print(f"  RMSE: {train_rmse:.4f}")
    print(f"  MAE:  {train_mae:.4f}")

    print("\nTesting set:")
    print(f"  R2 score: {test_r2_score:.4f}")
    print(f"  RMSE: {test_rmse:.4f}")
    print(f"  MAE:  {test_mae:.4f}")

    cv_score = cross_val_score(rm_model, X_train_scaled, y_train, cv=5, scoring='r2')

    print("\nCross validation (5 folds)")
    print(f"  R2 score: {cv_score}")
    print(f"  Cross val mean: {cv_score.mean()}")
    print(f"  Cross val std: {cv_score.std()}")

    rm_metrics = {
        "train_r2_score": train_r2_score,
        "test_r2_score": test_r2_score,
        "train_rmse": train_rmse,
        "test_rmse": test_rmse,
        "train_mae": train_mae,
        "test_mae": test_mae,
        "cv_score": cv_score

    }
        


    return rm_metrics

In [65]:
def save_model_artifact(model, scaler, label_encoder, feature_columns):
    print("\n" + "=" * 60)
    print("SAVING MODEL ARTIFACT")
    print("=" * 60)

    joblib.dump(model, "car_price_prediction.pkl")
    print("Car price prediction model saved")

    joblib.dump(scaler, "scaler_features.pkl")
    print("Scaler feature saved")

    joblib.dump(label_encoder, "label_encoder.pkl")
    print("Label feature encoder saved")

    joblib.dump(feature_columns, "feature_columns.pkl")
    print("Feature columns saved")

In [66]:
def save_rm_model_artifact(rm_model):
    print("\n" + "=" * 60)
    print("SAVING MODEL ARTIFACT")
    print("=" * 60)

    joblib.dump(rm_model, "rm_model_car_price_prediction.pkl")
    print("Car price prediction model saved")


In [88]:
def predict_car_price(year, make, model_name, transmission, condition):
    model = joblib.load("car_price_prediction.pkl")
    scaler = joblib.load("scaler_features.pkl")
    label_encoder = joblib.load("label_encoder.pkl")
    feature_columns = joblib.load("feature_columns.pkl")

    try:
        make_encoded = label_encoder['Make'].transform([make])[0]
        model_encoded = label_encoder['Model'].transform([model_name])[0]
        condition_encoded = label_encoder['Condition'].transform([condition])[0]
        transmission_encoded = label_encoder['Transmission'].transform([transmission])[0]
    except ValueError as e:
        return f"Unknown category {e}"
    features_dict ={
    'Year': year,
    'Make_encoded': make_encoded,
    'Model_encoded': model_encoded,
    'Transmission_encoded': transmission_encoded,
    'Condition_encoded': condition_encoded
    }

    features = np.array([[features_dict[col] for col in features_dict]])
    
    feature_scale = scaler.transform(features)

    predicted_price = model.predict(feature_scale)[0].item()

    return predicted_price

In [89]:
def test_prediction():
    print('\n' + '=' * 60)
    print("TEST CAR PREDICTION")
    print('=' * 60)
    
    # EXAMPLE 1
    price1 = predict_car_price(2015, "Toyota", "Camry", "Automatic", "Local Used")
    print('\n2015 Toyota Camry (Foreign Used Automatic)')
    print(f'₦{float(price1):,.2f}')  # Corrected from price2 to price1
    
    # EXAMPLE 2
    price2 = predict_car_price(2015, "Lexus", "RX 350", "Automatic", "Foreign Used")
    print('\n2012 Lexus RX 350 (Local Used Automatic)')
    print(f'₦{float(price2):,.2f}')
    
    # EXAMPLE 3
    price3 = predict_car_price(2015, "Honda", "Accord", "Automatic", "Local Used")
    print('\n2010 Honda Accord (Local Used Automatic)')
    print(f'₦{float(price3):,.2f}')



In [90]:
def main():
    file_path = "data/automation-analysis-cleaned-dataset.csv"

    df = load_and_explore_dataset(file_path)

    df_preprocessed, label_encoder = preprocessed_data(df)

    X, y, feature_columns = feature_data(df_preprocessed)

    X_train, X_test, y_train, y_test = split_data(X,y)

    X_train_scaled, X_test_scaled, scaler = scale_feature(X_train, X_test)

    model = train_model(X_train_scaled, y_train, feature_columns)

    metrics = evaluate_model(model, X_train_scaled, X_test_scaled, y_train, y_test)
    
    rm_model = rm_train_model(X_train_scaled, y_train, feature_columns)

    rm_metrics = rm_evaluate_model(rm_model, X_train_scaled, X_test_scaled, y_train, y_test)

    save_model_artifact(model, scaler, label_encoder, feature_columns)

    save_rm_model_artifact(rm_model)

    test_prediction()

In [91]:
if __name__ == "__main__":
    main()

LOAD AND EXPLORE DATASET
Shape of dataset
(1754, 7)

Checking for missing value:
Title           0
Make            0
Model           0
Year            0
Condition       0
Transmission    0
Price           0
dtype: int64

First five rows:
                                               Title           Make  \
0                  Mercedes Benz M Class 2015 Silver  Mercedes-Benz   
1                                  Toyota Camry 2013         Toyota   
2  Mercedes Benz C43 4Matic Sedan Awd (3.0L V6 9A...  Mercedes-Benz   
3  Mercedes Benz E350 Base 4Matic Wagon Awd (3.5L...  Mercedes-Benz   
4      Toyota Camry Le 4Dr Sedan (2.4L 4Cyl 4A) 2004         Toyota   

        Model  Year     Condition Transmission     Price  
0     M Class  2015  Foreign Used    Automatic  29109375  
1       Camry  2013    Local Used    Automatic   8650000  
2  C43 4Matic  2017    Local Used    Automatic  19500000  
3   E350 Base  2015  Foreign Used    Automatic  21500000  
4    Camry LE  2004  Foreign Used    Aut

C:\Users\user\Desktop\automobile_price_analysis_in_the_nigerian_used_car_market\venv\Lib\site-packages\sklearn\base.py:1403: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples,), for example using ravel().
  return fit_method(estimator, *args, **kwargs)



Model train successfully
  Year: 0.6749032538444839
  Model_encoded: 0.22580684743499793
  Make_encoded: 0.061552133347214516
  Condition_encoded: 0.03763394621154419
  Transmission_encoded: 0.00010381916175936928

RANDOM FOREST EVALUAING MODEL

RANDOM FOREST MODEL PERFORMANCE
TRAINING SET:
  R2 score: 0.9149
  RMSE: 3742133.5168
  MAE:  2253113.8334

Testing set:
  R2 score: 0.6919
  RMSE: 6172271.4925
  MAE:  3911302.9976


C:\Users\user\Desktop\automobile_price_analysis_in_the_nigerian_used_car_market\venv\Lib\site-packages\sklearn\base.py:1403: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples,), for example using ravel().
  return fit_method(estimator, *args, **kwargs)
C:\Users\user\Desktop\automobile_price_analysis_in_the_nigerian_used_car_market\venv\Lib\site-packages\sklearn\base.py:1403: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples,), for example using ravel().
  return fit_method(estimator, *args, **kwargs)
C:\Users\user\Desktop\automobile_price_analysis_in_the_nigerian_used_car_market\venv\Lib\site-packages\sklearn\base.py:1403: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples,), for example using ravel().
  return fit_method(estimator, *args, **kwargs)
C:\Users\user\De


Cross validation (5 folds)
  R2 score: [0.83288299 0.7248372  0.76053405 0.83809504 0.74907491]
  Cross val mean: 0.7810848371413985
  Cross val std: 0.04592162885402087

SAVING MODEL ARTIFACT
Car price prediction model saved
Scaler feature saved
Label feature encoder saved
Feature columns saved

SAVING MODEL ARTIFACT
Car price prediction model saved

TEST CAR PREDICTION

2015 Toyota Camry (Foreign Used Automatic)
₦19,361,902.64

2012 Lexus RX 350 (Local Used Automatic)
₦26,536,416.96

2010 Honda Accord (Local Used Automatic)
₦16,330,494.11


C:\Users\user\Desktop\automobile_price_analysis_in_the_nigerian_used_car_market\venv\Lib\site-packages\sklearn\utils\validation.py:2827: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
C:\Users\user\Desktop\automobile_price_analysis_in_the_nigerian_used_car_market\venv\Lib\site-packages\sklearn\utils\validation.py:2827: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
C:\Users\user\Desktop\automobile_price_analysis_in_the_nigerian_used_car_market\venv\Lib\site-packages\sklearn\utils\validation.py:2827: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
